In [1]:
import pandas as pd
from transformers import pipeline
import torch
import textwrap
# pip install transformer
# pip install tensorflow
# pip install torch
# pip install tf-keras
# pip install hf_xet to use CPU

In [2]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
dataset_file = "IMDB Dataset.csv"
numsamples = 5

In [4]:
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=model_name,
    tokenizer=model_name
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

C:\Users\kelly\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kelly\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sent

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [6]:
df = pd.read_csv(dataset_file)
df_positive = df[df['sentiment'] == 'positive'].sample(numsamples)
df_negative = df[df['sentiment'] == 'negative'].sample(numsamples)
sample_df = pd.concat([df_positive, df_negative])
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
label_map = {
    'label_0' : 'Negative',
    'label_1' : 'Neutral',
    'label_2' : 'Positive'
}

In [9]:
for index, row in sample_df.iterrows():
    review_text = row['review']
    actual_sentiment = row['sentiment'].capitalize()

    # snippet = textwrap.shorten(review_text, width=150, placeholder="...")
    wraptext = "\n".join(textwrap.wrap(review_text, width=150)).replace("<br /><br />", "")

    try:
        # Run the analysis. The pipeline handles truncation for the model.
        result = sentiment_pipeline(review_text, truncation=True)

        # Get the prediction
        predicted_label_raw = result[0]['label']
        predicted_sentiment = label_map.get(predicted_label_raw, predicted_label_raw)
        confidence = result[0]['score']


        print("\n" + "-" * 40)
        print(f"Review: \"{wraptext}\"")
        print(f"\nActual (from dataset): {actual_sentiment}")
        print(f"Model Prediction:      {predicted_sentiment} ({confidence:} confidence)")

        # Note if the model's prediction differs (e.g., "Neutral" vs "Positive")
        if predicted_sentiment.lower() != actual_sentiment.lower():
            print(f"Model prediction differs from dataset label.")

    except Exception as e:
        print(f"\nError analyzing review: {e}")

print("\n" + "=" * 40)
print("Analysis complete.")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



----------------------------------------
Review: "Very well done and spooky horror movie from poverty-row film company PRC who usually put out really cheesy films like DEVIL BAT or THE FLYING SERPENT.
German expatriate director Wisbar does wonders with a small budget and his studio-bound swamp set. Gaunt and ghoulish Charles Middleton is effective
as the Strangler."

Actual (from dataset): Positive
Model Prediction:      positive (0.8342152833938599 confidence)

----------------------------------------
Review: "This Alec Guinness starrer is a very good fun political satire of corporate industry, and a light eccentric character study as well.The
pacing is a bit slow for a comedy, and none of it is really rolling-on-the-floor type funny, except perhaps the sound effects for the experiments. But
it does have its amusing moments, and it is very deft in its execution. The big explosions segment is probably the most farcical element.<br /><br
/>The union procedures are quite droll, very rem